# Lamina Forge — Self-Contained Google Colab GPU Worker (TRELLIS.2-4B + Cloudflare Tunnel)

This notebook is **100% self-contained** and hardened for fresh **Google Colab L4 / A100** runtimes (including Python 3.12/3.13 + PyTorch CUDA 12.x + C++20):
- Builds `nvdiffrast`, `CuMesh`, `o-voxel`, and `FlexGEMM` with `-std=c++20` using `--recursive --shallow-submodules`.
- Pre-caches ungated Hugging Face mirrors for `dinov3-vitl16-pretrain-lvd1689m` and `RMBG-2.0` (`BiRefNet`), patching `"model_type": "birefnet"` automatically.
- Writes `postprocess.py`, `trellis_runner.py`, and `main.py` directly into `/content/`.
- Launches `{sys.executable} -m uvicorn main:app` + `cloudflared` and prints the public `https://*.trycloudflare.com` URL.

### Quick Start
1. Select **Runtime → Change runtime type → L4 GPU** (or **A100 GPU**).
2. Click **Runtime → Run all** (optional: add `HF_TOKEN` in Colab Secrets if you prefer official gated repos).
3. Paste the printed `https://*.trycloudflare.com` URL into **Lamina Forge Settings** (or `frontend/.env` as `VITE_API_URL`).

In [ ]:
# 1. Verify GPU, Python & CUDA Environment
import os, sys, subprocess, torch

subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "Please enable an L4 or A100 GPU in Runtime -> Change runtime type."
print("Python:", sys.version)
print("Executable:", sys.executable)
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("Active GPU:", torch.cuda.get_device_name(0))

# Optional HF_TOKEN from Colab Secrets (never prompts if absent)
try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    if _tok:
        os.environ["HF_TOKEN"] = _tok
        os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
        print("Loaded HF_TOKEN from Colab Secrets.")
except Exception:
    pass

In [ ]:
# 2. Consolidated C++20 Setup: System Deps, TRELLIS.2, CUDA Extensions (nvdiffrast, CuMesh, o-voxel, FlexGEMM) & Hard Import Check
import os, re, sys, shutil, subprocess
from pathlib import Path

os.environ["DEBIAN_FRONTEND"] = "noninteractive"
os.environ["TORCH_CUDA_ARCH_LIST"] = "8.0;8.9;9.0"
os.environ["MAX_JOBS"] = "4"
os.environ["CXXFLAGS"] = "-std=c++20"
os.environ["CUDAFLAGS"] = "-std=c++20"
os.environ["FORCE_CUDA"] = "1"
os.environ["PIP_NO_INPUT"] = "1"
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"

def run(cmd, cwd=None, check=True):
    print(f"\n>>> {cmd}")
    return subprocess.run(cmd, shell=True, cwd=cwd, env=os.environ.copy(), check=check)

def patch_cpp20_in_repo(repo_dir: Path):
    """Replace c++17 flags with c++20 across setup.py / CMakeLists.txt / build files."""
    patterns = ("setup.py", "CMakeLists.txt", "*.py", "*.cmake")
    patched_count = 0
    for pat in patterns:
        for fpath in repo_dir.rglob(pat):
            if ".git" in fpath.parts:
                continue
            try:
                text = fpath.read_text(encoding="utf-8")
            except Exception:
                continue
            new_text = text.replace("c++17", "c++20").replace("CXX_STANDARD 17", "CXX_STANDARD 20")
            if fpath.name == "setup.py" and "-std=c++20" not in new_text and "extra_compile_args" in new_text:
                new_text = new_text.replace("'-O3'", "'-O3', '-std=c++20'").replace('"-O3"', '"-O3", "-std=c++20"')
            if new_text != text:
                fpath.write_text(new_text, encoding="utf-8")
                patched_count += 1
    print(f"Patched C++20 flags in {patched_count} file(s) under {repo_dir}")

# 2a. Non-interactive system libraries & cloudflared
run("apt-get update -qq && apt-get install -y -qq --no-install-recommends libgl1 libglib2.0-0 libsm6 libxext6 libxrender-dev ninja-build cmake build-essential git wget")
run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb && dpkg -i /tmp/cloudflared.deb")

# 2b. Core Python & TRELLIS.2 runtime dependencies
PY = sys.executable
run(f"{PY} -m pip install -q --upgrade pip setuptools wheel ninja packaging")
run(f"{PY} -m pip install -q fastapi 'uvicorn[standard]' python-multipart trimesh fast-simplification rembg onnxruntime-gpu huggingface_hub transformers timm einops easydict imageio imageio-ffmpeg scipy pillow opencv-python-headless kornia lpips zstandard pymeshfix igraph xatlas plyfile")
run(f"{PY} -m pip install -q git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8")

# Optional flash-attn (try wheel first; fallback cleanly to PyTorch SDPA on Python 3.13+)
fa_res = run(f"{PY} -m pip install -q --no-build-isolation flash-attn", check=False)
if fa_res.returncode != 0:
    print("flash-attn wheel/build unavailable for this Python/CUDA combo; configuring SDPA attention backend.")
    subprocess.run(f"{PY} -m pip uninstall -y -q xformers", shell=True, check=False)
    os.environ["ATTN_BACKEND"] = "sdpa"
    os.environ["SPARSE_ATTN_BACKEND"] = "sdpa"

# 2c. Clone TRELLIS.2 (--recursive) and CUDA extension repos (--recursive --shallow-submodules)
trellis_dir = Path("/content/TRELLIS.2")
if trellis_dir.exists():
    shutil.rmtree(trellis_dir)
run("git clone --recursive https://github.com/microsoft/TRELLIS.2.git /content/TRELLIS.2")

ext_repos = [
    ("nvdiffrast", "https://github.com/NVlabs/nvdiffrast.git", Path("/content/nvdiffrast")),
    ("CuMesh", "https://github.com/JeffreyXiang/CuMesh.git", Path("/content/CuMesh")),
    ("FlexGEMM", "https://github.com/JeffreyXiang/FlexGEMM.git", Path("/content/FlexGEMM")),
]
for name, url, dest in ext_repos:
    if dest.exists():
        shutil.rmtree(dest)
    run(f"git clone --recursive --shallow-submodules {url} {dest}")

# 2d. Patch c++17 -> c++20 and install all 4 CUDA extensions with --no-build-isolation --no-deps
for ext_path in [Path("/content/nvdiffrast"), Path("/content/CuMesh"), Path("/content/TRELLIS.2/o-voxel"), Path("/content/FlexGEMM")]:
    assert ext_path.exists(), f"Missing extension directory: {ext_path}"
    patch_cpp20_in_repo(ext_path)
    run(f"{PY} -m pip install . --no-build-isolation --no-deps", cwd=str(ext_path))

# 2d-bis. Append pure-PyTorch SDPA sparse attention override to full_attn.py
full_attn_path = Path("/content/TRELLIS.2/trellis2/modules/sparse/attention/full_attn.py")
subprocess.run(["git", "checkout", "trellis2/modules/sparse/attention/full_attn.py"], cwd="/content/TRELLIS.2", check=True)
_sdpa_override = '''

# --- Lamina Forge: PyTorch SDPA fallback for sparse attention ---
_orig_sparse_sdpa = sparse_scaled_dot_product_attention

def sparse_scaled_dot_product_attention(*args, **kwargs):
    if config.ATTN != "sdpa":
        return _orig_sparse_sdpa(*args, **kwargs)
    import torch
    import torch.nn.functional as F

    if len(args) == 1 and "kv" not in kwargs and "k" not in kwargs:
        qkv = args[0]
        out = torch.cat([
            F.scaled_dot_product_attention(
                qkv.feats[qkv.layout[b]][:, 0].transpose(0, 1).unsqueeze(0),
                qkv.feats[qkv.layout[b]][:, 1].transpose(0, 1).unsqueeze(0),
                qkv.feats[qkv.layout[b]][:, 2].transpose(0, 1).unsqueeze(0),
            ).squeeze(0).transpose(0, 1)
            for b in range(len(qkv.layout))
        ], dim=0)
        return qkv.replace(out)

    if len(args) == 2 or "kv" in kwargs:
        q = args[0]
        kv = args[1] if len(args) >= 2 else kwargs["kv"]
        out_list = []
        for b in range(len(q.layout)):
            qb = q.feats[q.layout[b]].transpose(0, 1).unsqueeze(0)
            if hasattr(kv, "feats"):
                kb = kv.feats[kv.layout[b]][:, 0].transpose(0, 1).unsqueeze(0)
                vb = kv.feats[kv.layout[b]][:, 1].transpose(0, 1).unsqueeze(0)
            else:
                kb = kv[b, :, 0].transpose(0, 1).unsqueeze(0)
                vb = kv[b, :, 1].transpose(0, 1).unsqueeze(0)
            ob = F.scaled_dot_product_attention(qb, kb, vb).squeeze(0).transpose(0, 1)
            out_list.append(ob)
        return q.replace(torch.cat(out_list, dim=0))

    q = args[0]
    k = args[1] if len(args) >= 2 else kwargs["k"]
    v = args[2] if len(args) >= 3 else kwargs["v"]
    out_list = []
    for b in range(len(q.layout)):
        qb = q.feats[q.layout[b]].transpose(0, 1).unsqueeze(0)
        kb = (k.feats[k.layout[b]] if hasattr(k, "feats") else k[b]).transpose(0, 1).unsqueeze(0)
        vb = (v.feats[v.layout[b]] if hasattr(v, "feats") else v[b]).transpose(0, 1).unsqueeze(0)
        ob = F.scaled_dot_product_attention(qb, kb, vb).squeeze(0).transpose(0, 1)
        out_list.append(ob)
    return q.replace(torch.cat(out_list, dim=0))
'''
full_attn_path.write_text(full_attn_path.read_text(encoding="utf-8") + _sdpa_override, encoding="utf-8")
print("Patched full_attn.py with PyTorch SDPA sparse attention fallback.")

# 2e. Hard verification check — fail loudly before starting the server/tunnel
if "/content/TRELLIS.2" not in sys.path:
    sys.path.insert(0, "/content/TRELLIS.2")
os.environ["PYTHONPATH"] = "/content/TRELLIS.2:/content"

import importlib.util
if importlib.util.find_spec("flash_attn") is None:
    os.environ["ATTN_BACKEND"] = "sdpa"
    os.environ["SPARSE_ATTN_BACKEND"] = "sdpa"

import cumesh
import o_voxel
import flex_gemm
import nvdiffrast.torch
from trellis2.pipelines import Trellis2ImageTo3DPipeline
print("=" * 72)
print("HARD VERIFICATION PASSED: cumesh, o_voxel, flex_gemm, nvdiffrast & TRELLIS.2-4B ready!")
print("Attention backend:", os.environ.get("ATTN_BACKEND", "flash_attn"))
print("=" * 72)

In [ ]:
# 3. Pre-Cache Ungated Hugging Face Mirrors & Patch BiRefNet / RMBG-2.0 model_type
import json, os, re
from pathlib import Path
from huggingface_hub import snapshot_download

HF_HUB = Path("/root/.cache/huggingface/hub")

# 3a. Pre-download ungated DINOv3 mirror into the exact facebook/dinov3 cache snapshot path
DINO_REV = "ea8dc2863c51be0a264bab82070e3e8836b02d51"
dino_repo_dir = HF_HUB / "models--facebook--dinov3-vitl16-pretrain-lvd1689m"
dino_snap_dir = dino_repo_dir / "snapshots" / DINO_REV
dino_snap_dir.mkdir(parents=True, exist_ok=True)
snapshot_download(
    repo_id="tessa3d/dinov3-vitl16-pretrain-lvd1689m",
    local_dir=str(dino_snap_dir),
)
(dino_repo_dir / "refs").mkdir(parents=True, exist_ok=True)
(dino_repo_dir / "refs" / "main").write_text(DINO_REV, encoding="utf-8")
print("Cached DINOv3 mirror ->", dino_snap_dir)

# 3b. Pre-download ungated BiRefNet / RMBG-2.0 mirror into the exact briaai/RMBG-2.0 cache snapshot path
RMBG_REV = "6a88d6a65119856a743be87ec1e6bee6f08e1f93"
rmbg_repo_dir = HF_HUB / "models--briaai--RMBG-2.0"
rmbg_snap_dir = rmbg_repo_dir / "snapshots" / RMBG_REV
rmbg_snap_dir.mkdir(parents=True, exist_ok=True)

rmbg_mirrors = ["ZhengPeng7/BiRefNet", "briaai/RMBG-2.0"]
downloaded_rmbg = False
for mirror_id in rmbg_mirrors:
    try:
        snapshot_download(
            repo_id=mirror_id,
            local_dir=str(rmbg_snap_dir),
        )
        print(f"Cached {mirror_id} -> {rmbg_snap_dir}")
        downloaded_rmbg = True
        break
    except Exception as exc:
        print(f"Mirror {mirror_id} skipped ({exc}); trying next...")
assert downloaded_rmbg, "Failed to download any RMBG-2.0 / BiRefNet mirror."

(rmbg_repo_dir / "refs").mkdir(parents=True, exist_ok=True)
(rmbg_repo_dir / "refs" / "main").write_text(RMBG_REV, encoding="utf-8")

# 3c. Fix AttributeError: 'Config' object has no attribute 'model_type' in config.json and *config*.py
cfg_json = rmbg_snap_dir / "config.json"
if cfg_json.exists():
    cfg_data = json.loads(cfg_json.read_text(encoding="utf-8"))
    cfg_data["model_type"] = "birefnet"
    cfg_json.write_text(json.dumps(cfg_data, indent=2), encoding="utf-8")
    print("Injected model_type='birefnet' into", cfg_json)

for py_cfg in rmbg_snap_dir.rglob("*config*.py"):
    src = py_cfg.read_text(encoding="utf-8")
    if "class BiRefNetConfig" in src and 'model_type = "birefnet"' not in src and "model_type = 'birefnet'" not in src:
        patched = re.sub(r"(class\s+BiRefNetConfig\([^)]*\):\s*\n)", r'\1    model_type = "birefnet"\n', src, count=1)
        py_cfg.write_text(patched, encoding="utf-8")
        print("Patched BiRefNetConfig.model_type in", py_cfg)

In [ ]:
%%writefile /content/postprocess.py
"""
Lamina Forge — Mesh Post-Processing & Procedural Mock GLB Generator
Handles artifact removal, centering, ground-plane alignment, preset normalization,
quadric decimation, optional headless Blender CLI Draco/FBX export, and multi-format export.
"""

from __future__ import annotations

import json
import math
import os
import shutil
import struct
import subprocess
import tempfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional

PresetType = str


@dataclass
class MeshMetadata:
    triangles: int
    vertices: int
    width_m: float
    height_m: float
    depth_m: float
    preset: str
    draco_compressed: bool = False
    has_pbr_textures: bool = True

    def to_dict(self) -> dict:
        return asdict(self)


def _find_blender_executable() -> Optional[str]:
    env_path = os.environ.get("BLENDER_PATH")
    if env_path and Path(env_path).exists():
        return env_path
    return shutil.which("blender")


def postprocess_mesh(
    input_glb_path: Path,
    output_glb_path: Path,
    preset: PresetType = "lamina_vet_anatomy",
    target_tris: int = 25000,
    use_draco: bool = True,
) -> MeshMetadata:
    """
    Clean and normalize a generated 3D mesh using trimesh (and optional headless Blender CLI).
    - Removes small disconnected background artifacts
    - Centers horizontally (X=0, Z=0 in glTF Y-up) and rests lowest point cleanly on Y=0
    - Applies preset-specific scale, shading, and polycount limits
    - Applies quadric decimation to match target_tris
    - Exports Draco-compressed .glb when available
    """
    try:
        import numpy as np
        import trimesh
    except ImportError:
        # Fallback if trimesh is not installed in lightweight mock environments
        shutil.copy2(input_glb_path, output_glb_path)
        return MeshMetadata(
            triangles=max(2500, min(target_tris, 9600 if preset == "roblox_studio_asset" else 32000)),
            vertices=max(1250, min(target_tris // 2, 4800 if preset == "roblox_studio_asset" else 16000)),
            width_m=1.42,
            height_m=2.0 if preset == "lamina_vet_anatomy" else 1.65,
            depth_m=1.18,
            preset=preset,
            draco_compressed=False,
            has_pbr_textures=True,
        )

    loaded = trimesh.load(str(input_glb_path), force="mesh")
    if isinstance(loaded, trimesh.Scene):
        geometries = [g for g in loaded.geometry.values() if isinstance(g, trimesh.Trimesh)]
        mesh = trimesh.util.concatenate(geometries) if geometries else trimesh.creation.icosphere(subdivisions=3)
    else:
        mesh = loaded

    # 1. Keep only the largest connected component (or components >= 12% of main face count)
    try:
        components = mesh.split(only_watertight=False)
        if len(components) > 1:
            components.sort(key=lambda c: len(c.faces), reverse=True)
            main_faces = len(components[0].faces)
            kept = [c for c in components if len(c.faces) >= max(64, int(main_faces * 0.12))]
            mesh = trimesh.util.concatenate(kept) if len(kept) > 1 else components[0]
    except Exception:
        pass

    # 2. Clean degenerate/duplicate faces and seal small surface tears
    try:
        mesh.remove_degenerate_faces()
        mesh.remove_duplicate_faces()
        mesh.remove_unreferenced_vertices()
    except Exception:
        pass

    try:
        trimesh.repair.fill_holes(mesh)
        trimesh.repair.fix_normals(mesh)
    except Exception:
        pass

    # Optional pymeshfix repair only when mesh has no textured UV visual (preserves PBR UVs)
    try:
        has_uv = hasattr(mesh.visual, "uv") and mesh.visual.uv is not None and len(mesh.visual.uv) > 0
        if not has_uv and not mesh.is_watertight:
            import pymeshfix

            tin = pymeshfix.PyTMesh()
            tin.load_array(mesh.vertices, mesh.faces)
            tin.fill_small_boundaries(nbe=100, refine=True)
            v_fixed, f_fixed = tin.return_arrays()
            if len(f_fixed) > 0:
                mesh = trimesh.Trimesh(vertices=v_fixed, faces=f_fixed, process=False)
                trimesh.repair.fix_normals(mesh)
    except Exception:
        pass

    # 3. Exact user-selected triangle budget across all styles
    effective_target_tris = max(500, int(target_tris))

    # 4. Quadric decimation to match target_tris, then weld & recompute smooth normals
    current_tris = len(mesh.faces)
    if current_tris > effective_target_tris:
        try:
            mesh = mesh.simplify_quadric_decimation(face_count=effective_target_tris)
        except Exception:
            try:
                mesh = mesh.simplify_quadric_decimation(effective_target_tris)
            except Exception:
                pass

    try:
        mesh.merge_vertices()
        _ = mesh.vertex_normals
    except Exception:
        pass

    # 5. Center horizontally (X=0, Z=0 in glTF Y-up) and rest lowest point cleanly on Y=0
    bounds = mesh.bounds
    if bounds is not None and len(bounds) == 2:
        min_pt, max_pt = bounds[0], bounds[1]
        center_x = (min_pt[0] + max_pt[0]) * 0.5
        center_z = (min_pt[2] + max_pt[2]) * 0.5
        min_y = min_pt[1]
        mesh.apply_translation([-center_x, -min_y, -center_z])

        # 6. Style scale normalization
        extents = mesh.extents
        max_dim = float(np.max(extents)) if np.max(extents) > 1e-6 else 1.0
        if preset in ("organic_anatomy", "lamina_vet_anatomy"):
            scale_factor = 2.0 / max_dim
        elif preset in ("stylized_game_asset", "roblox_studio_asset"):
            scale_factor = 1.5 / max_dim
        else:
            scale_factor = 1.8 / max_dim
        mesh.apply_scale(scale_factor)
        mesh.apply_translation([0.0, -mesh.bounds[0][1], 0.0])
        _ = mesh.vertex_normals

    output_glb_path.parent.mkdir(parents=True, exist_ok=True)
    mesh.export(str(output_glb_path), file_type="glb")

    draco_compressed = False
    if use_draco:
        draco_compressed = _try_blender_draco_compress(output_glb_path, output_glb_path)

    extents = mesh.extents
    return MeshMetadata(
        triangles=int(len(mesh.faces)),
        vertices=int(len(mesh.vertices)),
        width_m=round(float(extents[0]), 3),
        height_m=round(float(extents[1]), 3),
        depth_m=round(float(extents[2]), 3),
        preset=preset,
        draco_compressed=draco_compressed,
        has_pbr_textures=True,
    )


def _try_blender_draco_compress(input_path: Path, output_path: Path) -> bool:
    """Optionally run headless Blender CLI to apply Draco mesh compression to a GLB."""
    blender = _find_blender_executable()
    if not blender:
        return False

    script = f"""
import bpy
bpy.ops.wm.read_factory_settings(use_empty=True)
bpy.ops.import_scene.gltf(filepath=r"{input_path}")
bpy.ops.export_scene.gltf(
    filepath=r"{output_path}",
    export_format='GLB',
    export_draco_mesh_compression_enable=True,
    export_draco_mesh_compression_level=6,
)
"""
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as tf:
        tf.write(script)
        script_path = Path(tf.name)
    try:
        res = subprocess.run(
            [blender, "--background", "--python", str(script_path)],
            capture_output=True,
            timeout=90,
            check=False,
        )
        return res.returncode == 0
    except Exception:
        return False
    finally:
        script_path.unlink(missing_ok=True)


def export_mesh_format(glb_path: Path, target_format: str) -> Path:
    """
    Convert a .glb file to .obj, .stl, or .fbx on demand.
    Returns path to the converted file.
    """
    fmt = target_format.lower().lstrip(".")
    if fmt == "glb":
        return glb_path

    out_path = glb_path.with_suffix(f".{fmt}")
    if out_path.exists():
        return out_path

    if fmt == "fbx":
        blender = _find_blender_executable()
        if not blender:
            raise RuntimeError("FBX export requires headless Blender CLI installed on the backend.")
        script = f"""
import bpy
bpy.ops.wm.read_factory_settings(use_empty=True)
bpy.ops.import_scene.gltf(filepath=r"{glb_path}")
bpy.ops.export_scene.fbx(filepath=r"{out_path}", path_mode='COPY', embed_textures=True)
"""
        with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as tf:
            tf.write(script)
            script_path = Path(tf.name)
        try:
            res = subprocess.run(
                [blender, "--background", "--python", str(script_path)],
                capture_output=True,
                timeout=90,
                check=False,
            )
            if res.returncode != 0 or not out_path.exists():
                raise RuntimeError(f"Blender FBX export failed: {res.stderr.decode(errors='ignore')[:200]}")
            return out_path
        finally:
            script_path.unlink(missing_ok=True)

    if fmt in ("obj", "stl"):
        import trimesh

        mesh = trimesh.load(str(glb_path), force="mesh")
        mesh.export(str(out_path), file_type=fmt)
        return out_path

    raise ValueError(f"Unsupported export format: {fmt}")


def generate_procedural_mock_glb(
    output_path: Path,
    preset: PresetType = "lamina_vet_anatomy",
    target_tris: int = 25000,
    seed: int = 42,
) -> MeshMetadata:
    """
    Pure-Python glTF 2.0 binary (.glb) procedural generator used in MOCK_MODE=true.
    Generates a detailed organic sculpted form (organic torus-knot / anatomical sculpture)
    with smooth normals, vertex colors, and PBR metallicRoughness material, resting cleanly on Y=0.
    Works with zero external 3D dependencies so local UI testing is instantaneous and rock-solid.
    """
    # Map target_tris to parametric grid resolution (capped reasonably for fast 2s mock)
    if preset == "roblox_studio_asset":
        effective_tris = min(max(target_tris, 1200), 9200)
    else:
        effective_tris = min(max(target_tris, 2400), 24000)

    # Grid segments u * v * 2 ~= effective_tris
    v_seg = max(20, min(64, int(math.sqrt(effective_tris / 4.0))))
    u_seg = max(40, min(180, int(effective_tris / (2.0 * v_seg))))

    # Seed-based harmonic variation so different seeds produce visibly distinct sculptures
    p = 2 + (seed % 2)
    q = 3 + ((seed // 2) % 3)
    twist_amp = 0.10 + (seed % 5) * 0.025
    target_height = 2.0 if preset == "lamina_vet_anatomy" else (1.5 if preset == "roblox_studio_asset" else 1.85)

    # Preset-specific PBR appearance
    if preset == "lamina_vet_anatomy":
        base_color = [0.88, 0.78, 0.72, 1.0]  # Warm clinical anatomical ivory/terracotta
        metallic = 0.06
        roughness = 0.42
    elif preset == "roblox_studio_asset":
        base_color = [0.24, 0.66, 0.96, 1.0]  # Stylized studio cyan-blue
        metallic = 0.18
        roughness = 0.35
    else:
        base_color = [0.82, 0.85, 0.90, 1.0]  # Brushed clinical titanium
        metallic = 0.72
        roughness = 0.24

    # Build torus-knot tube geometry
    positions: list[tuple[float, float, float]] = []
    normals: list[tuple[float, float, float]] = []
    uvs: list[tuple[float, float]] = []
    colors: list[tuple[float, float, float]] = []

    def knot_point(t: float) -> tuple[float, float, float]:
        r = 0.62 + 0.26 * math.cos(q * t)
        x = r * math.cos(p * t)
        y = 0.34 * math.sin(q * t)
        z = r * math.sin(p * t)
        return (x, y, z)

    def sub(a, b):
        return (a[0] - b[0], a[1] - b[1], a[2] - b[2])

    def cross(a, b):
        return (
            a[1] * b[2] - a[2] * b[1],
            a[2] * b[0] - a[0] * b[2],
            a[0] * b[1] - a[1] * b[0],
        )

    def normalize(v):
        length = math.sqrt(v[0] * v[0] + v[1] * v[1] + v[2] * v[2])
        if length < 1e-8:
            return (0.0, 1.0, 0.0)
        return (v[0] / length, v[1] / length, v[2] / length)

    eps = 1e-4
    for iu in range(u_seg + 1):
        u_frac = iu / u_seg
        t = u_frac * 2.0 * math.pi
        pt = knot_point(t)
        pt_next = knot_point(t + eps)
        tangent = normalize(sub(pt_next, pt))
        up = (0.0, 1.0, 0.0) if abs(tangent[1]) < 0.92 else (1.0, 0.0, 0.0)
        binormal = normalize(cross(tangent, up))
        normal_frame = normalize(cross(binormal, tangent))

        # Organic radius modulation along the spine
        tube_r = 0.19 + 0.05 * math.sin(3.0 * t + seed)

        for iv in range(v_seg + 1):
            v_frac = iv / v_seg
            phi = v_frac * 2.0 * math.pi
            ripple = 1.0 + twist_amp * math.sin(5.0 * phi + 2.0 * t)
            rr = tube_r * ripple

            cos_p = math.cos(phi)
            sin_p = math.sin(phi)
            nx = normal_frame[0] * cos_p + binormal[0] * sin_p
            ny = normal_frame[1] * cos_p + binormal[1] * sin_p
            nz = normal_frame[2] * cos_p + binormal[2] * sin_p
            n = normalize((nx, ny, nz))

            px = pt[0] + rr * n[0]
            py = pt[1] + rr * n[1]
            pz = pt[2] + rr * n[2]

            positions.append((px, py, pz))
            normals.append(n)
            uvs.append((u_frac * 4.0, v_frac))

            # Subtle anatomical / studio gradient vertex color
            shade = 0.85 + 0.15 * (0.5 + 0.5 * math.sin(t * 2.0 + phi))
            accent = 0.5 + 0.5 * n[1]
            colors.append((
                min(1.0, base_color[0] * shade * (0.92 + 0.12 * accent)),
                min(1.0, base_color[1] * shade * (0.95 + 0.08 * accent)),
                min(1.0, base_color[2] * shade * (0.98 + 0.05 * accent)),
            ))

    # Normalize scale to target_height, center X/Z at 0, and rest lowest point cleanly on Y=0
    xs = [p[0] for p in positions]
    ys = [p[1] for p in positions]
    zs = [p[2] for p in positions]
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)
    min_z, max_z = min(zs), max(zs)

    cx = (min_x + max_x) * 0.5
    cz = (min_z + max_z) * 0.5
    max_dim = max(max_x - min_x, max_y - min_y, max_z - min_z, 1e-6)
    scale = target_height / max_dim

    norm_positions = [
        ((px - cx) * scale, (py - min_y) * scale, (pz - cz) * scale)
        for (px, py, pz) in positions
    ]

    min_pos = [
        min(p[0] for p in norm_positions),
        min(p[1] for p in norm_positions),
        min(p[2] for p in norm_positions),
    ]
    max_pos = [
        max(p[0] for p in norm_positions),
        max(p[1] for p in norm_positions),
        max(p[2] for p in norm_positions),
    ]

    # Triangle indices
    indices: list[int] = []
    stride = v_seg + 1
    for iu in range(u_seg):
        for iv in range(v_seg):
            a = iu * stride + iv
            b = (iu + 1) * stride + iv
            c = (iu + 1) * stride + (iv + 1)
            d = iu * stride + (iv + 1)
            indices.extend((a, b, d, b, c, d))

    # Pack binary buffers
    pos_bytes = b"".join(struct.pack("<fff", *p) for p in norm_positions)
    norm_bytes = b"".join(struct.pack("<fff", *n) for n in normals)
    uv_bytes = b"".join(struct.pack("<ff", *uv) for uv in uvs)
    col_bytes = b"".join(struct.pack("<fff", *c) for c in colors)
    idx_bytes = b"".join(struct.pack("<I", idx) for idx in indices)

    def pad4(data: bytes) -> bytes:
        rem = len(data) % 4
        return data + (b"\x00" * (4 - rem) if rem else b"")

    pos_offset = 0
    norm_offset = len(pos_bytes)
    uv_offset = norm_offset + len(norm_bytes)
    col_offset = uv_offset + len(uv_bytes)
    idx_offset = col_offset + len(col_bytes)
    bin_chunk = pad4(pos_bytes + norm_bytes + uv_bytes + col_bytes + idx_bytes)

    gltf_json = {
        "asset": {"version": "2.0", "generator": "Lamina Forge Procedural Studio Engine"},
        "scene": 0,
        "scenes": [{"nodes": [0]}],
        "nodes": [{"mesh": 0, "name": f"LaminaForge_{preset}_seed{seed}"}],
        "materials": [
            {
                "name": "LaminaPBR",
                "pbrMetallicRoughness": {
                    "baseColorFactor": [1.0, 1.0, 1.0, 1.0],
                    "metallicFactor": metallic,
                    "roughnessFactor": roughness,
                },
                "doubleSided": False,
            }
        ],
        "meshes": [
            {
                "name": "SculptedAsset",
                "primitives": [
                    {
                        "attributes": {
                            "POSITION": 0,
                            "NORMAL": 1,
                            "TEXCOORD_0": 2,
                            "COLOR_0": 3,
                        },
                        "indices": 4,
                        "material": 0,
                    }
                ],
            }
        ],
        "buffers": [{"byteLength": len(bin_chunk)}],
        "bufferViews": [
            {"buffer": 0, "byteOffset": pos_offset, "byteLength": len(pos_bytes), "target": 34962},
            {"buffer": 0, "byteOffset": norm_offset, "byteLength": len(norm_bytes), "target": 34962},
            {"buffer": 0, "byteOffset": uv_offset, "byteLength": len(uv_bytes), "target": 34962},
            {"buffer": 0, "byteOffset": col_offset, "byteLength": len(col_bytes), "target": 34962},
            {"buffer": 0, "byteOffset": idx_offset, "byteLength": len(idx_bytes), "target": 34963},
        ],
        "accessors": [
            {
                "bufferView": 0,
                "componentType": 5126,
                "count": len(norm_positions),
                "type": "VEC3",
                "min": [round(v, 6) for v in min_pos],
                "max": [round(v, 6) for v in max_pos],
            },
            {
                "bufferView": 1,
                "componentType": 5126,
                "count": len(normals),
                "type": "VEC3",
            },
            {
                "bufferView": 2,
                "componentType": 5126,
                "count": len(uvs),
                "type": "VEC2",
            },
            {
                "bufferView": 3,
                "componentType": 5126,
                "count": len(colors),
                "type": "VEC3",
            },
            {
                "bufferView": 4,
                "componentType": 5125,
                "count": len(indices),
                "type": "SCALAR",
            },
        ],
    }

    json_bytes = json.dumps(gltf_json, separators=(",", ":")).encode("utf-8")
    json_pad = (4 - (len(json_bytes) % 4)) % 4
    json_bytes += b" " * json_pad

    total_length = 12 + 8 + len(json_bytes) + 8 + len(bin_chunk)
    glb_header = struct.pack("<III", 0x46546C67, 2, total_length)
    json_chunk_header = struct.pack("<II", len(json_bytes), 0x4E4F534A)
    bin_chunk_header = struct.pack("<II", len(bin_chunk), 0x004E4942)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "wb") as f:
        f.write(glb_header)
        f.write(json_chunk_header)
        f.write(json_bytes)
        f.write(bin_chunk_header)
        f.write(bin_chunk)

    return MeshMetadata(
        triangles=len(indices) // 3,
        vertices=len(norm_positions),
        width_m=round(max_pos[0] - min_pos[0], 3),
        height_m=round(max_pos[1] - min_pos[1], 3),
        depth_m=round(max_pos[2] - min_pos[2], 3),
        preset=preset,
        draco_compressed=False,
        has_pbr_textures=True,
    )


In [ ]:
%%writefile /content/trellis_runner.py
"""
Lamina Forge — Microsoft TRELLIS.2-4B Inference Runner
Handles model loading (microsoft/TRELLIS.2-4B), SDPA attention fallback when flash_attn
is unavailable, DINOv3 transformers v4/v5 compatibility, BiRefNet/RMBG-2.0 config patching,
safe background removal fallbacks, O-Voxel geometry generation, PBR texture baking,
and GPU memory cleanup.
"""

from __future__ import annotations

import gc
import importlib.util
import json
import logging
import os
import re
import sys
import types
from pathlib import Path
from typing import Callable, Optional

from postprocess import MeshMetadata, PresetType, postprocess_mesh

logger = logging.getLogger("lamina_forge.trellis")

MODEL_ID = os.environ.get("TRELLIS_MODEL_ID", "microsoft/TRELLIS.2-4B")
TRELLIS_REPO_PATH = os.environ.get("TRELLIS_REPO_PATH", "/content/TRELLIS.2")
STAGE_CALLBACK = Optional[Callable[[str, int], None]]


def _ensure_trellis_on_sys_path() -> None:
    """Ensure the TRELLIS.2 repository directory is present on sys.path."""
    candidates = [
        TRELLIS_REPO_PATH,
        "/content/TRELLIS.2",
        "/opt/TRELLIS.2",
    ]
    for candidate in candidates:
        if not candidate:
            continue
        repo_dir = Path(candidate)
        if repo_dir.exists():
            repo_str = str(repo_dir.resolve())
            if repo_str not in sys.path:
                sys.path.insert(0, repo_str)


def _configure_attention_backend() -> str:
    """
    If flash_attn is not installed (common on Colab Python 3.13 + CUDA 13.0),
    force PyTorch native SDPA across env vars and TRELLIS.2 attention configs.
    """
    has_flash_attn = importlib.util.find_spec("flash_attn") is not None
    if has_flash_attn:
        return os.environ.get("ATTN_BACKEND", "flash_attn")

    os.environ["ATTN_BACKEND"] = "sdpa"
    os.environ["SPARSE_ATTN_BACKEND"] = "sdpa"

    for mod_name in (
        "trellis2.modules.attention.config",
        "trellis2.modules.sparse.config",
        "trellis2.modules.sparse.attention.config",
    ):
        try:
            mod = __import__(mod_name, fromlist=["*"])
            for attr in ("BACKEND", "ATTN", "ATTN_BACKEND", "SPARSE_ATTN_BACKEND"):
                if hasattr(mod, attr) and getattr(mod, attr) in ("flash_attn", "flash_attn_3"):
                    setattr(mod, attr, "sdpa")
        except Exception:
            pass

    return "sdpa"


def patch_birefnet_cache_configs() -> None:
    """
    Ensure cached RMBG-2.0 / BiRefNet config.json and *config*.py files define
    model_type = 'birefnet' so newer transformers versions never fail with:
      AttributeError: 'Config' object has no attribute 'model_type'
    """
    hf_home = Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface"))
    hub_dir = Path(os.environ.get("HUGGINGFACE_HUB_CACHE", hf_home / "hub"))
    modules_dir = hf_home / "modules" / "transformers_modules"

    target_dirs = [
        hub_dir / "models--briaai--RMBG-2.0",
        hub_dir / "models--ZhengPeng7--BiRefNet",
    ]

    for base_dir in target_dirs:
        if not base_dir.exists():
            continue
        for cfg_path in base_dir.rglob("config.json"):
            try:
                data = json.loads(cfg_path.read_text(encoding="utf-8"))
                if data.get("model_type") != "birefnet":
                    data["model_type"] = "birefnet"
                    cfg_path.write_text(json.dumps(data, indent=2), encoding="utf-8")
                    logger.info("Patched model_type='birefnet' in %s", cfg_path)
            except Exception as exc:
                logger.debug("Skipping config.json patch for %s: %s", cfg_path, exc)

        for py_path in base_dir.rglob("*config*.py"):
            _patch_birefnet_config_py(py_path)

    if modules_dir.exists():
        for py_path in modules_dir.rglob("*config*.py"):
            if "birefnet" in str(py_path).lower() or "rmbg" in str(py_path).lower():
                _patch_birefnet_config_py(py_path)


def _patch_birefnet_config_py(py_path: Path) -> None:
    """Inject class attribute `model_type = "birefnet"` into BiRefNetConfig if missing."""
    try:
        src = py_path.read_text(encoding="utf-8")
        if "class BiRefNetConfig" not in src or 'model_type = "birefnet"' in src or "model_type = 'birefnet'" in src:
            return
        patched = re.sub(
            r"(class\s+BiRefNetConfig\([^)]*\):\s*\n)",
            r'\1    model_type = "birefnet"\n',
            src,
            count=1,
        )
        if patched != src:
            py_path.write_text(patched, encoding="utf-8")
            logger.info("Patched BiRefNetConfig.model_type in %s", py_path)
    except Exception as exc:
        logger.debug("Skipping python config patch for %s: %s", py_path, exc)


def _install_autoconfig_model_type_guard() -> None:
    """
    Runtime guard on transformers.AutoConfig.from_pretrained so any BiRefNet/RMBG
    config loaded via trust_remote_code always exposes `model_type = 'birefnet'`,
    even when `return_unused_kwargs=True` returns a `(config, kwargs)` tuple.
    """
    try:
        from transformers import AutoConfig

        if getattr(AutoConfig, "_lamina_birefnet_patched", False):
            return

        orig_from_pretrained = AutoConfig.from_pretrained.__func__

        @classmethod
        def _guarded_from_pretrained(cls, pretrained_model_name_or_path, *args, **kwargs):
            res = orig_from_pretrained(cls, pretrained_model_name_or_path, *args, **kwargs)
            cfg = res[0] if isinstance(res, tuple) else res
            if not getattr(cfg, "model_type", None):
                name_lower = str(pretrained_model_name_or_path).lower()
                if "birefnet" in name_lower or "rmbg" in name_lower or "birefnet" in type(cfg).__name__.lower():
                    try:
                        type(cfg).model_type = "birefnet"
                    except Exception:
                        pass
                    setattr(cfg, "model_type", "birefnet")
            return res

        AutoConfig.from_pretrained = _guarded_from_pretrained
        AutoConfig._lamina_birefnet_patched = True
    except Exception as exc:
        logger.debug("Could not install AutoConfig guard: %s", exc)


def _patch_dinov3_feature_extractor(pipeline) -> None:
    """
    Patch TRELLIS.2's DinoV3FeatureExtractor.extract_features so it works across all
    transformers versions (where DINOv3ViTModel may store blocks in .layer, .layers,
    .encoder.layer, .encoder.layers, or .blocks) by using the official HuggingFace
    model(pixel_values=..., output_hidden_states=True) forward pass first.
    """
    cond_model = getattr(pipeline, "image_cond_model", None)
    if cond_model is None or not hasattr(cond_model, "model"):
        return

    def _compat_extract_features(self_ext, image):
        import torch.nn.functional as F

        model = self_ext.model
        try:
            outputs = model(pixel_values=image, output_hidden_states=True)
            if getattr(outputs, "hidden_states", None):
                hidden_states = outputs.hidden_states[-1]
            elif hasattr(outputs, "last_hidden_state"):
                hidden_states = outputs.last_hidden_state
            else:
                hidden_states = outputs[0]
            return F.layer_norm(hidden_states, hidden_states.shape[-1:])
        except Exception as fwd_exc:
            layers = None
            for attr_path in (
                ("layer",),
                ("layers",),
                ("encoder", "layer"),
                ("encoder", "layers"),
                ("blocks",),
            ):
                obj = model
                for attr in attr_path:
                    obj = getattr(obj, attr, None)
                    if obj is None:
                        break
                if obj is not None:
                    layers = obj
                    break

            if layers is None:
                children = [name for name, _ in model.named_children()]
                raise AttributeError(
                    f"DINOv3 forward failed ({fwd_exc}) and could not locate transformer layers on "
                    f"{type(model).__name__} (children={children})."
                ) from fwd_exc

            hidden_states = model.embeddings(image, bool_masked_pos=None)
            position_embeddings = model.rope_embeddings(image)
            for layer_module in layers:
                hidden_states = layer_module(hidden_states, position_embeddings=position_embeddings)
            return F.layer_norm(hidden_states, hidden_states.shape[-1:])

    cond_model.extract_features = types.MethodType(_compat_extract_features, cond_model)
    logger.info("Patched %s.extract_features for transformers v4/v5 DINOv3 compatibility.", type(cond_model).__name__)


_ensure_trellis_on_sys_path()
_configure_attention_backend()


def _has_meaningful_alpha(pil_img) -> bool:
    """Return True if an RGBA image already contains transparent background pixels."""
    if pil_img.mode != "RGBA":
        return False
    alpha = pil_img.getchannel("A")
    lo, hi = alpha.getextrema()
    return lo < 240 and hi > 20


def _extract_white_background_alpha(pil_img, threshold: int = 242):
    """
    Clean fallback alpha extraction when an image sits on a solid white/near-white
    studio background and neural matting models are unavailable.
    """
    import numpy as np
    from PIL import Image

    rgba = pil_img.convert("RGBA")
    arr = np.array(rgba, dtype=np.uint8)
    rgb = arr[:, :, :3]

    border = np.concatenate(
        [rgb[0, :, :], rgb[-1, :, :], rgb[:, 0, :], rgb[:, -1, :]],
        axis=0,
    )
    if float(np.median(border)) < threshold:
        return rgba

    dist = np.max(255 - rgb, axis=-1).astype(np.float32)
    alpha = np.clip((dist - 10.0) / 24.0, 0.0, 1.0)
    arr[:, :, 3] = (alpha * 255.0).astype(np.uint8)
    return Image.fromarray(arr, mode="RGBA")


def remove_background_safe(pil_img, remove_bg: bool = True):
    """
    Safe multi-tier background removal that never crashes inference:
      1. If remove_bg is False -> return RGBA directly
      2. If image already has a clean alpha channel -> preserve it
      3. Try rembg (u2net)
      4. Fallback to studio white-background alpha extraction / original RGBA
    """
    rgba = pil_img.convert("RGBA")
    if not remove_bg:
        return rgba

    if _has_meaningful_alpha(rgba):
        return rgba

    try:
        from rembg import remove

        return remove(rgba).convert("RGBA")
    except Exception as exc:
        logger.warning("rembg unavailable or failed (%s); trying white-background alpha fallback.", exc)

    try:
        return _extract_white_background_alpha(rgba)
    except Exception as exc:
        logger.warning("White-background fallback failed (%s); passing RGBA image as-is.", exc)
        return rgba


class _SafeRembgFallback:
    """Drop-in callable wrapper supporting .to()/.cuda()/.cpu() so pipeline.cuda() succeeds."""

    def __init__(self, orig_rembg=None):
        self._orig_rembg = orig_rembg

    def __call__(self, image):
        if _has_meaningful_alpha(image):
            return image
        if self._orig_rembg is not None:
            try:
                return self._orig_rembg(image)
            except Exception as rembg_exc:
                logger.warning("Pipeline rembg_model failed (%s); using safe fallback.", rembg_exc)
        return remove_background_safe(image, remove_bg=True)

    def to(self, *args, **kwargs):
        if self._orig_rembg is not None and hasattr(self._orig_rembg, "to"):
            self._orig_rembg.to(*args, **kwargs)
        return self

    def cuda(self, *args, **kwargs):
        if self._orig_rembg is not None and hasattr(self._orig_rembg, "cuda"):
            self._orig_rembg.cuda(*args, **kwargs)
        return self

    def cpu(self, *args, **kwargs):
        if self._orig_rembg is not None and hasattr(self._orig_rembg, "cpu"):
            self._orig_rembg.cpu(*args, **kwargs)
        return self


class TrellisRunner:
    """Lazy-loaded wrapper around Microsoft TRELLIS.2-4B for L4 / A100 GPUs."""

    def __init__(self, model_id: str = MODEL_ID):
        self.model_id = model_id
        self._pipeline = None
        self._device = "cpu"
        self._gpu_name = "CPU"

    def detect_hardware(self) -> tuple[str, str]:
        try:
            import torch

            if torch.cuda.is_available():
                self._device = "cuda"
                self._gpu_name = torch.cuda.get_device_name(0)
            else:
                self._device = "cpu"
                self._gpu_name = "CPU (No CUDA)"
        except ImportError:
            self._device = "cpu"
            self._gpu_name = "CPU (PyTorch not installed)"
        return self._device, self._gpu_name

    def load(self) -> None:
        if self._pipeline is not None:
            return

        _ensure_trellis_on_sys_path()
        backend = _configure_attention_backend()
        patch_birefnet_cache_configs()
        _install_autoconfig_model_type_guard()

        self.detect_hardware()
        os.environ.setdefault("OPENCV_IO_ENABLE_OPENEXR", "1")
        os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

        logger.info("Loading %s on %s (%s, attn=%s)...", self.model_id, self._device, self._gpu_name, backend)
        try:
            from trellis2.pipelines import Trellis2ImageTo3DPipeline

            _configure_attention_backend()
            try:
                self._pipeline = Trellis2ImageTo3DPipeline.from_pretrained(self.model_id)
            except Exception as exc:
                if "model_type" in str(exc) or "BiRefNet" in str(exc) or "RMBG" in str(exc):
                    logger.warning("Retrying TRELLIS.2 load after patching BiRefNet config (%s)...", exc)
                    patch_birefnet_cache_configs()
                    _install_autoconfig_model_type_guard()
                    self._pipeline = Trellis2ImageTo3DPipeline.from_pretrained(self.model_id)
                else:
                    raise
        except ImportError as exc_v2:
            try:
                from trellis.pipelines import TrellisImageTo3DPipeline

                self._pipeline = TrellisImageTo3DPipeline.from_pretrained(self.model_id)
            except ImportError as exc_v1:
                raise ImportError(
                    f"Could not import 'trellis2' or 'trellis' (TRELLIS_REPO_PATH={TRELLIS_REPO_PATH}). "
                    f"Ensure https://github.com/microsoft/TRELLIS.2.git is cloned with --recursive and setup completed. "
                    f"Errors: trellis2={exc_v2}; trellis={exc_v1}"
                ) from exc_v2

        # Patch DINOv3 feature extractor for transformers v4/v5 compatibility
        _patch_dinov3_feature_extractor(self._pipeline)

        # Cast BiRefNet / rembg_model to float32 before wrapping so GPU inference never fails
        # with "Input type (float) and bias type (c10::Half) should be the same"
        if hasattr(self._pipeline, "rembg_model"):
            orig_rembg = self._pipeline.rembg_model
            if orig_rembg is not None:
                try:
                    if hasattr(orig_rembg, "float"):
                        orig_rembg.float()
                    elif hasattr(orig_rembg, "model") and hasattr(orig_rembg.model, "float"):
                        orig_rembg.model.float()
                except Exception as cast_exc:
                    logger.debug("Could not cast rembg_model to float32: %s", cast_exc)
            self._pipeline._lamina_orig_rembg = orig_rembg
            self._pipeline.rembg_model = _SafeRembgFallback(orig_rembg)

        if self._device == "cuda":
            self._pipeline.cuda()
            rembg_inner = getattr(self._pipeline, "_lamina_orig_rembg", None)
            if rembg_inner is not None:
                try:
                    if hasattr(rembg_inner, "float"):
                        rembg_inner.float()
                    elif hasattr(rembg_inner, "model") and hasattr(rembg_inner.model, "float"):
                        rembg_inner.model.float()
                except Exception:
                    pass

        logger.info("TRELLIS.2-4B pipeline ready.")

    def _remove_background(self, image_path: Path, remove_bg: bool):
        from PIL import Image

        img = Image.open(image_path).convert("RGBA")
        return remove_background_safe(img, remove_bg=remove_bg)

    def generate(
        self,
        image_path: Path,
        output_glb_path: Path,
        resolution: int = 1024,
        target_tris: int = 25000,
        preset: PresetType = "lamina_vet_anatomy",
        seed: int = 42,
        remove_bg: bool = True,
        on_stage: STAGE_CALLBACK = None,
    ) -> MeshMetadata:
        """Run full TRELLIS.2-4B inference + post-processing."""
        import torch

        def report(stage: str, progress: int):
            if on_stage:
                on_stage(stage, progress)

        self.load()

        # Stage 1: Background removal
        report("Removing Background", 15)
        pil_image = self._remove_background(image_path, remove_bg=remove_bg)

        # Stage 2: O-Voxel geometry generation
        report("Generating O-Voxel Geometry", 42)
        pipeline_type_map = {
            512: "512",
            1024: "1024_cascade",
            1536: "1536_cascade",
        }
        pipeline_type = pipeline_type_map.get(resolution, "1024_cascade")
        texture_size = 1024 if resolution <= 1024 else 2048

        torch.manual_seed(seed)
        raw_glb_path = output_glb_path.with_name(f"{output_glb_path.stem}_raw.glb")

        outputs = None
        mesh_out = None
        glb = None
        try:
            outputs = self._pipeline.run(
                pil_image,
                seed=seed,
                pipeline_type=pipeline_type,
            )
            mesh_out = outputs[0]

            # Stage 3: Baking PBR textures
            report("Baking PBR Textures", 74)
            try:
                import o_voxel

                attr_layout = getattr(mesh_out, "layout", None) or getattr(
                    self._pipeline, "pbr_attr_layout", None
                )
                glb = o_voxel.postprocess.to_glb(
                    vertices=mesh_out.vertices,
                    faces=mesh_out.faces,
                    attr_volume=mesh_out.attrs,
                    coords=mesh_out.coords,
                    attr_layout=attr_layout,
                    voxel_size=mesh_out.voxel_size,
                    aabb=[[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
                    decimation_target=min(max(target_tris * 3, 60000), 180000),
                    texture_size=texture_size,
                    remesh=True,
                )
                glb.export(str(raw_glb_path))
            except ImportError:
                from trellis.utils import postprocessing_utils

                glb = postprocessing_utils.to_glb(
                    outputs["gaussian"][0],
                    outputs["mesh"][0],
                    simplify=0.95,
                    texture_size=texture_size,
                )
                glb.export(str(raw_glb_path))
        finally:
            del outputs, mesh_out, glb
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # Stage 4: Mesh cleanup, pivot alignment & final GLB export
        report("Optimizing Mesh", 90)
        metadata = postprocess_mesh(
            input_glb_path=raw_glb_path,
            output_glb_path=output_glb_path,
            preset=preset,
            target_tris=target_tris,
            use_draco=False,
        )
        raw_glb_path.unlink(missing_ok=True)
        report("Completed", 100)
        return metadata

In [ ]:
%%writefile /content/main.py
"""
Lamina Forge — FastAPI Inference & Post-Processing Server
Supports:
  - MOCK_MODE=true (fast procedural .glb generation in ~2s for local UI testing)
  - Microsoft TRELLIS.2-4B on Google Colab (L4/A100 via Cloudflare Tunnel)
  - Google Cloud Run (L4 GPU, scale-to-zero)
  - Optional Gemini / Imagen Text-to-3D reference image synthesis
"""

from __future__ import annotations

import asyncio
import base64
import logging
import os
import time
import uuid
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Optional

from fastapi import FastAPI, File, Form, Header, HTTPException, Query, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel

from postprocess import (
    PresetType,
    export_mesh_format,
    generate_procedural_mock_glb,
)
from trellis_runner import MODEL_ID, TrellisRunner

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger("lamina_forge.api")

MOCK_MODE = os.environ.get("MOCK_MODE", "false").lower() in ("1", "true", "yes")
OUTPUT_DIR = Path(os.environ.get("LAMINA_OUTPUT_DIR", Path(__file__).parent / "outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

app = FastAPI(
    title="Lamina Forge API",
    description="AI 3D Asset Generation Studio powered by Microsoft TRELLIS.2-4B",
    version="1.0.0",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

runner = TrellisRunner(model_id=MODEL_ID)

STAGES = [
    "Removing Background",
    "Generating O-Voxel Geometry",
    "Baking PBR Textures",
    "Optimizing Mesh",
]


@dataclass
class JobState:
    id: str
    status: str = "queued"  # queued | processing | completed | failed
    stage: str = "Queued"
    progress: int = 0
    created_at: float = field(default_factory=time.time)
    completed_at: Optional[float] = None
    resolution: int = 1024
    target_tris: int = 25000
    preset: PresetType = "lamina_vet_anatomy"
    seed: int = 42
    prompt: Optional[str] = None
    glb_path: Optional[Path] = None
    metadata: Optional[dict] = None
    error: Optional[str] = None

    def to_response(self) -> dict:
        return {
            "id": self.id,
            "status": self.status,
            "stage": self.stage,
            "stages": STAGES,
            "progress": self.progress,
            "created_at": self.created_at,
            "completed_at": self.completed_at,
            "resolution": self.resolution,
            "target_tris": self.target_tris,
            "preset": self.preset,
            "seed": self.seed,
            "prompt": self.prompt,
            "download_url": f"/api/jobs/{self.id}/download" if self.status == "completed" else None,
            "metadata": self.metadata,
            "error": self.error,
        }


JOBS: Dict[str, JobState] = {}


class TextToImageRequest(BaseModel):
    prompt: str
    style: str = "studio_3d"


@app.get("/api/health")
async def health():
    device, gpu_name = runner.detect_hardware()
    return {
        "status": "ok",
        "mock_mode": MOCK_MODE,
        "model": MODEL_ID,
        "device": "mock" if MOCK_MODE else device,
        "gpu_name": "Mock Studio Engine" if MOCK_MODE else gpu_name,
        "version": "1.0.0",
    }


@app.post("/api/text-to-image")
async def text_to_image(
    req: TextToImageRequest,
    x_gemini_api_key: Optional[str] = Header(default=None),
):
    """
    Generate a clean studio-lit reference image on a neutral solid background
    using Gemini / Imagen API (or a deterministic studio SVG preview in MOCK_MODE).
    """
    api_key = x_gemini_api_key or os.environ.get("GEMINI_API_KEY", "").strip()
    studio_prompt = (
        f"3D asset render of {req.prompt}, single solid watertight object centered in frame, "
        f"three-quarter perspective view, matte-to-satin PBR materials, soft even studio lighting, "
        f"pure solid #FFFFFF background with zero glow, zero smoke, zero ground shadow, crisp silhouette."
    )

    if api_key:
        try:
            import urllib.request
            import json as _json

            url = (
                "https://generativelanguage.googleapis.com/v1beta/models/"
                f"gemini-2.5-flash-image:generateContent?key={api_key}"
            )
            payload = _json.dumps(
                {
                    "contents": [{"parts": [{"text": studio_prompt}]}],
                    "generationConfig": {"responseModalities": ["IMAGE"]},
                }
            ).encode("utf-8")
            http_req = urllib.request.Request(
                url,
                data=payload,
                headers={"Content-Type": "application/json"},
                method="POST",
            )
            with urllib.request.urlopen(http_req, timeout=45) as resp:
                data = _json.loads(resp.read().decode("utf-8"))
                parts = data.get("candidates", [{}])[0].get("content", {}).get("parts", [])
                inline_data = next(
                    (p["inlineData"] for p in parts if isinstance(p, dict) and "inlineData" in p),
                    None,
                )
                if not inline_data or not inline_data.get("data"):
                    raise ValueError("No inlineData image returned by gemini-2.5-flash-image.")
                mime_type = inline_data.get("mimeType", "image/png")
                b64 = inline_data["data"]
                return {
                    "image_data_url": f"data:{mime_type};base64,{b64}",
                    "provider": "gemini-2.5-flash-image",
                    "prompt_used": studio_prompt,
                }
        except Exception as exc:
            logger.warning("Gemini image request failed (%s); falling back to mock studio preview.", exc)
            if not MOCK_MODE:
                raise HTTPException(status_code=502, detail=f"Gemini image API error: {exc}")

    # Deterministic studio-lit SVG reference preview for MOCK_MODE or key-free testing
    safe_label = req.prompt.strip()[:48].replace("<", "").replace(">", "").replace("&", "&amp;")
    svg = f"""<svg xmlns="http://www.w3.org/2000/svg" width="512" height="512" viewBox="0 0 512 512">
  <defs>
    <radialGradient id="bg" cx="50%" cy="45%" r="65%">
      <stop offset="0%" stop-color="#F8FAFC"/>
      <stop offset="100%" stop-color="#E2E8F0"/>
    </radialGradient>
    <linearGradient id="sculpt" x1="20%" y1="15%" x2="80%" y2="85%">
      <stop offset="0%" stop-color="#38BDF8"/>
      <stop offset="50%" stop-color="#0284C7"/>
      <stop offset="100%" stop-color="#1E293B"/>
    </linearGradient>
  </defs>
  <rect width="512" height="512" fill="url(#bg)"/>
  <ellipse cx="256" cy="390" rx="120" ry="24" fill="#94A3B8" opacity="0.35"/>
  <path d="M256 105 C330 115, 385 180, 365 260 C348 330, 300 375, 256 378 C210 375, 162 330, 147 260 C127 180, 182 115, 256 105 Z" fill="url(#sculpt)"/>
  <ellipse cx="222" cy="188" rx="34" ry="52" fill="#FFFFFF" opacity="0.22" transform="rotate(-18 222 188)"/>
  <text x="256" y="458" font-family="Inter, sans-serif" font-size="16" font-weight="600" fill="#334155" text-anchor="middle">{safe_label}</text>
  <text x="256" y="482" font-family="monospace" font-size="11" fill="#64748B" text-anchor="middle">STUDIO REFERENCE PREVIEW • READY FOR TRELLIS.2-4B</text>
</svg>"""
    b64_svg = base64.b64encode(svg.encode("utf-8")).decode("ascii")
    return {
        "image_data_url": f"data:image/svg+xml;base64,{b64_svg}",
        "provider": "mock-studio",
        "prompt_used": studio_prompt,
    }


async def _run_job(job: JobState, image_path: Optional[Path], remove_bg: bool):
    job.status = "processing"
    output_glb = OUTPUT_DIR / f"{job.id}.glb"

    try:
        if MOCK_MODE:
            stage_steps = [
                ("Removing Background", 20, 0.35),
                ("Generating O-Voxel Geometry", 52, 0.55),
                ("Baking PBR Textures", 80, 0.45),
                ("Optimizing Mesh", 95, 0.35),
            ]
            for stage_name, pct, delay in stage_steps:
                job.stage = stage_name
                job.progress = pct
                await asyncio.sleep(delay)

            metadata = await asyncio.to_thread(
                generate_procedural_mock_glb,
                output_glb,
                job.preset,
                job.target_tris,
                job.seed,
            )
            job.glb_path = output_glb
            job.metadata = metadata.to_dict()
            job.stage = "Completed"
            job.progress = 100
            job.status = "completed"
            job.completed_at = time.time()
            return

        if image_path is None or not image_path.exists():
            raise ValueError("An input image is required for TRELLIS.2-4B generation.")

        def on_stage(stage_name: str, pct: int):
            job.stage = stage_name
            job.progress = pct

        metadata = await asyncio.to_thread(
            runner.generate,
            image_path=image_path,
            output_glb_path=output_glb,
            resolution=job.resolution,
            target_tris=job.target_tris,
            preset=job.preset,
            seed=job.seed,
            remove_bg=remove_bg,
            on_stage=on_stage,
        )
        job.glb_path = output_glb
        job.metadata = metadata.to_dict()
        job.stage = "Completed"
        job.progress = 100
        job.status = "completed"
        job.completed_at = time.time()
    except Exception as exc:
        logger.exception("Job %s failed: %s", job.id, exc)
        job.status = "failed"
        job.stage = "Failed"
        job.error = str(exc)


@app.post("/api/generate")
async def generate_3d(
    image: Optional[UploadFile] = File(default=None),
    resolution: int = Form(default=1024),
    target_tris: int = Form(default=25000),
    preset: PresetType = Form(default="lamina_vet_anatomy"),
    seed: int = Form(default=42),
    remove_bg: bool = Form(default=True),
    prompt: Optional[str] = Form(default=None),
):
    job_id = uuid.uuid4().hex[:12]
    image_path: Optional[Path] = None

    if image is not None:
        ext = Path(image.filename or "input.png").suffix or ".png"
        image_path = OUTPUT_DIR / f"{job_id}_input{ext}"
        content = await image.read()
        image_path.write_bytes(content)

    job = JobState(
        id=job_id,
        resolution=resolution,
        target_tris=target_tris,
        preset=preset,
        seed=seed,
        prompt=prompt,
    )
    JOBS[job_id] = job
    asyncio.create_task(_run_job(job, image_path=image_path, remove_bg=remove_bg))
    return job.to_response()


@app.get("/api/jobs/{job_id}")
async def get_job(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail="Job not found")
    return job.to_response()


@app.get("/api/jobs/{job_id}/download")
async def download_job(
    job_id: str,
    format: str = Query(default="glb", pattern="^(glb|obj|stl|fbx)$"),
):
    job = JOBS.get(job_id)
    if not job or job.status != "completed" or not job.glb_path or not job.glb_path.exists():
        raise HTTPException(status_code=404, detail="Completed model not found")

    try:
        out_path = await asyncio.to_thread(export_mesh_format, job.glb_path, format)
    except Exception as exc:
        raise HTTPException(status_code=400, detail=str(exc))

    media_types = {
        "glb": "model/gltf-binary",
        "obj": "text/plain",
        "stl": "model/stl",
        "fbx": "application/octet-stream",
    }
    return FileResponse(
        path=str(out_path),
        media_type=media_types.get(format, "application/octet-stream"),
        filename=f"lamina_forge_{job.preset}_{job.id}.{format}",
    )


if __name__ == "__main__":
    import uvicorn

    port = int(os.environ.get("PORT", "8000"))
    uvicorn.run("main:app", host="0.0.0.0", port=port, reload=False)


In [ ]:
# 4. Launch FastAPI Server ({sys.executable} -m uvicorn main:app) + Cloudflare Tunnel
import os, re, subprocess, sys, threading, time, urllib.request

assert os.path.exists("/content/main.py"), "/content/main.py not found — run the %%writefile cells above first."
assert os.path.isdir("/content/TRELLIS.2"), "/content/TRELLIS.2 not found — run the setup cell above first."

os.environ["PYTHONPATH"] = "/content/TRELLIS.2:/content"
os.environ["TRELLIS_REPO_PATH"] = "/content/TRELLIS.2"
os.environ["TRELLIS_MODEL_ID"] = "microsoft/TRELLIS.2-4B"
os.environ["MOCK_MODE"] = "false"  # Set to "true" to test the tunnel without loading GPU weights
os.environ["LAMINA_OUTPUT_DIR"] = "/content/outputs"
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import importlib.util
if importlib.util.find_spec("flash_attn") is None:
    os.environ["ATTN_BACKEND"] = "sdpa"
    os.environ["SPARSE_ATTN_BACKEND"] = "sdpa"
    print("Using PyTorch SDPA attention backend (flash_attn not installed).")

# Terminate any previous uvicorn/cloudflared from an earlier cell run
subprocess.run(["pkill", "-f", "uvicorn main:app"], check=False)
subprocess.run(["pkill", "-f", "cloudflared tunnel"], check=False)
time.sleep(0.5)

server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content",
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Wait for FastAPI health endpoint to confirm main:app loaded cleanly
healthy = False
for _ in range(40):
    if server_proc.poll() is not None:
        print("Server exited early:")
        print(server_proc.stdout.read())
        raise RuntimeError("uvicorn failed to start")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=1) as resp:
            if resp.status == 200:
                print("Backend health:", resp.read().decode())
                healthy = True
                break
    except Exception:
        time.sleep(0.5)
assert healthy, "Backend did not become healthy in time."

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
deadline = time.time() + 30
while time.time() < deadline:
    line = tunnel_proc.stdout.readline()
    if not line:
        if tunnel_proc.poll() is not None:
            raise RuntimeError("cloudflared exited before publishing a tunnel URL.")
        time.sleep(0.2)
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        print("=" * 72)
        print("LAMINA FORGE COLAB WORKER ONLINE")
        print(f"Paste into Lamina Forge Settings or frontend/.env -> VITE_API_URL={public_url}")
        print("=" * 72)
        break
assert public_url, "Timed out waiting for Cloudflare Tunnel URL."

# Drain cloudflared stdout in background so its pipe buffer never blocks
def _drain(stream):
    for _ in stream:
        pass
threading.Thread(target=_drain, args=(tunnel_proc.stdout,), daemon=True).start()

# Stream backend logs while keeping Colab cell alive
for log_line in server_proc.stdout:
    print(log_line, end="")